# Lignes de champ magnétique

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import lib

# Figures interactives (zoom, déplacement) dans Jupyter quand l'extension ipympl est
# installée ; ailleurs, cette instruction est sans effet
try:
    get_ipython().run_line_magic("matplotlib", "widget")
except Exception:
    pass

# Résolution réduite pour que les calculs restent rapides en ligne
h = 1e-3# m, pas de la grille des circuits magnétiques
pas = 0.02# pas de la grille de la spire, du dipôle et de l'aimant
lib.LARGEUR = 6# largeur des figures à l'écran
lib.AXES = True# axes gradués

## Spire parcourue par un courant

Spire circulaire d'axe vertical, vue en coupe dans un plan contenant son axe.

In [ ]:
R = 1.0# rayon de la spire
r_fil = 0.08# rayon du fil dessiné
I = 1/(np.pi*R**2)# intensité donnant un moment magnétique I pi R² = 1

# moment vers le haut : le courant vient vers nous à gauche et s'éloigne à droite
lib.figure_revolution(lambda X, Y: lib.flux_spire(X, Y, R, I),
                      lambda ax: lib.dessine_fils(ax, [(-R, 0, r_fil, 1), (R, 0, r_fil, -1)]),
                      pas=pas)
plt.show()

## Dipôle magnétique

La flèche au centre indique le sens du moment magnétique.

In [ ]:
r_dipole = 0.4# rayon du disque qui représente le dipôle

lib.figure_revolution(lib.flux_dipole, lambda ax: lib.dessine_dipole(ax, r_dipole),
                      rayon_objet=r_dipole, n_niveaux=round(lib.R_EQ/r_dipole), pas=pas)
plt.show()

## Aimant droit

Aimant cylindrique aimanté uniformément vers le haut, vu en coupe dans un plan contenant son axe.

In [ ]:
R = 0.4# rayon de l'aimant
L = 2.0# hauteur de l'aimant
M = 1/(np.pi*R**2*L)# aimantation donnant un moment magnétique M pi R² L = 1

lib.figure_revolution(lambda X, Y: lib.flux_aimant(X, Y, R, L, M),
                      lambda ax: lib.dessine_aimant(ax, R, L), pas=pas)
plt.show()

## Circuit magnétique sans entrefer

Circuit de section rectangulaire, supposé très long perpendiculairement à la figure. Le bobinage entoure la branche de gauche.

In [ ]:
a = 0.10# m, largeur extérieure du circuit
b = 0.12# m, hauteur extérieure du circuit
d = 0.02# m, largeur des branches du circuit
Ni = 500# A, nombre de spires x intensité
mu_r = 1e5# perméabilité relative du fer

fer, fils = lib.cadre_bobine(a, b, d, 0, Ni)
lib.figure_circuit(fer, fils, mu_r, h=h)
plt.show()

## Circuit magnétique avec entrefer

Même circuit, avec un entrefer dans la branche de droite. La cellule compare le champ au centre de l'entrefer à la formule du cours.

In [ ]:
a = 0.10# m, largeur extérieure du circuit
b = 0.12# m, hauteur extérieure du circuit
d = 0.02# m, largeur des branches du circuit
e = 5e-3# m, épaisseur de l'entrefer
Ni = 500# A, nombre de spires x intensité
mu_r = 1e5# perméabilité relative du fer

fer, fils = lib.cadre_bobine(a, b, d, e, Ni)
x, y, A, Bx, By = lib.figure_circuit(fer, fils, mu_r, h=h)
plt.show()

l = 2*(a - d) + 2*(b - d) - e# m, longueur de la ligne moyenne dans le fer
i, j = np.argmin(abs(y)), np.argmin(abs(x - (a/2 - d/2)))
print(f"B au centre de l'entrefer (simulation)   : {np.hypot(Bx[i, j], By[i, j])*1e3:.1f} mT")
print(f"mu0 N i / (e + l/mu_r) (fer de mu_r fini) : {lib.mu0*Ni/(e + l/mu_r)*1e3:.1f} mT")
print(f"mu0 N i / e            (formule du cours) : {lib.mu0*Ni/e*1e3:.1f} mT")

## Électroaimant

Pièce en U retournée, bobinée sur sa branche horizontale, face à un barreau : le circuit comporte deux entrefers.

In [ ]:
a = 0.10# m, largeur du U et du barreau
b = 0.08# m, hauteur du U
d = 0.02# m, largeur des branches du U et épaisseur du barreau
e = 5e-3# m, épaisseur de chaque entrefer
Ni = 500# A, nombre de spires x intensité
mu_r = 1e5# perméabilité relative du fer

fer, fils, y_entrefers = lib.electroaimant(a, b, d, e, Ni)
x, y, A, Bx, By = lib.figure_circuit(fer, fils, mu_r, h=h, y_fleches=y_entrefers)
plt.show()

# les deux entrefers sont en série
l = 2*(a - d) + 2*b# m, longueur de la ligne moyenne dans le fer
i, j = np.argmin(abs(y - y_entrefers)), np.argmin(abs(x - (a/2 - d/2)))
print(f"B au centre d'un entrefer (simulation)     : {np.hypot(Bx[i, j], By[i, j])*1e3:.1f} mT")
print(f"mu0 N i / (2e + l/mu_r) (fer de mu_r fini) : {lib.mu0*Ni/(2*e + l/mu_r)*1e3:.1f} mT")
print(f"mu0 N i / 2e            (formule du cours) : {lib.mu0*Ni/(2*e)*1e3:.1f} mT")